In [108]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

In [109]:
df = pd.read_csv('../../../integrated/data/transcript_portfolio_profile.csv')

In [110]:
df2 = df.copy()

In [111]:
df2.head()

,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,profile_missing
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,0
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26,1
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0


In [112]:
df2.isna().sum()

customer_id              0
event                    0
time                     0
offer_id            138953
amount              167184
event_reward        272955
offer_type          138953
offer_reward        138953
difficulty          138953
duration            138953
channels            138953
web                 138953
email               138953
mobile              138953
social              138953
gender               33749
age                  33749
income               33749
became_member_on     33749
profile_missing          0
dtype: int64

In [113]:
df2['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33182
Name: count, dtype: int64

# 시간 기준 없이 전체 전환율 다시 구해보기
- 기준: offer received -> offer completed 기준으로 정의
- transaction은 오퍼와 직접 연결이 어려워 본 분석에서는 제외
- 이후 프로모션(offer_type), 채널별 전환율 비교 진행 예정
- 일단 프로모션 수신이랑 완료를 기준으로 전체 전환율구할 예정 그다음 duration 기준으로 전환율 판단해보기

In [114]:
df2['duration'].value_counts()

duration
7.0     69969
5.0     37289
10.0    33860
3.0     14305
4.0     11761
Name: count, dtype: int64

일단 프로모션을 수신한사람이 프로모션을 얼마나 완료했나부터 다시 해보기

1. 프로모션수신이랑 완료를 고객+오퍼 기준으로 분리해서 보기

In [115]:
received = df2[df2['event'] == 'offer received'][['customer_id', 'offer_id']]

In [116]:
completed = df2[df2['event'] == 'offer completed'][['customer_id', 'offer_id']]

In [117]:
received

,customer_id,offer_id
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0
...,...,...
257635,d087c473b4d247ccb0abfef59ba12b0e,ae264e3637204a6fb9bb56bc8210ddfd
257636,cb23b66c56f64b109d673d5e56574529,2906b810c7d4411798c6938adc9daaa5
257637,6d5f3a774f3d4714ab0c092238f3a1d7,2298d6c36e964ae4a3e7e9706d1fb8c2
257638,9dc1421481194dcd9400aec7c9ae6366,ae264e3637204a6fb9bb56bc8210ddfd


In [118]:
completed

,customer_id,offer_id
12658,9fa9ae8f57894cc9a3b8a9bbe0fc1b2f,2906b810c7d4411798c6938adc9daaa5
12672,fe97aa22dd3e48c8b143116a8403dd52,fafdcd668e3743c1bb461111dcafc2a4
12679,629fc02d56414d91bca360decdfa9288,9b98b8c7a33c4b65b9aebfe6a799e6d9
12692,676506bad68e4161b9bbaffeb039626b,ae264e3637204a6fb9bb56bc8210ddfd
12697,8f7dd3b2afe14c078eb4f6e6fe4ba97d,4d5c57ea9a6940dd891ad53e9dbe8da0
...,...,...
306078,0c027f5f34dd4b9eba0a25785c611273,2298d6c36e964ae4a3e7e9706d1fb8c2
306100,a6f84f4e976f44508c358cc9aba6d2b3,2298d6c36e964ae4a3e7e9706d1fb8c2
306109,b895c57e8cd047a8872ce02aa54759d6,fafdcd668e3743c1bb461111dcafc2a4
306112,8431c16f8e1d440880db371a68f82dd0,fafdcd668e3743c1bb461111dcafc2a4


In [119]:
received.duplicated().sum()

np.int64(12989)

In [120]:
completed.duplicated().sum()

np.int64(4186)

2. 같은 고객이 같은 오퍼를 여러번 기록한것도 있어 이런 경우 제거하는게 좋다고 판단

In [121]:
received = received.drop_duplicates()

In [122]:
completed = completed.drop_duplicates()

In [123]:
received.duplicated().sum()

np.int64(0)

In [124]:
completed.duplicated().sum()

np.int64(0)

중복 제거는 완료 본격적으로 전환율 분석

In [125]:
# 프로모션수신을 받은건 유지하고 완료한 사람이 있으면 유지 고객이 이 프로모션을 받고 완료했는가 merge컬럼 생성
conv = received.merge(completed, on=['customer_id', 'offer_id'], how='left', indicator=True)

In [126]:
# both가 수신을받고 오퍼까지 완료한 고객 left_only가 수신을 했지만 오퍼를 완료하지 않은 고객
conv

,customer_id,offer_id,_merge
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,both
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7,left_only
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,left_only
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4,left_only
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0,left_only
...,...,...,...
63283,670626b55bfb4ba39c85b27cc7cca527,0b1e1539f2cc45b7b9fa7c272da2e1d7,both
63284,a57890c3bbb7463e9018abb7fecadb15,5a8bc65990b245e5a138643cd4eb9837,left_only
63285,d087c473b4d247ccb0abfef59ba12b0e,ae264e3637204a6fb9bb56bc8210ddfd,both
63286,6d5f3a774f3d4714ab0c092238f3a1d7,2298d6c36e964ae4a3e7e9706d1fb8c2,left_only


In [127]:
# 숫자로 수신과 오퍼를받고 완료한 고객 1, 수신했지만 오퍼를 완료안한 고객 0으로  

In [128]:
conv['converted'] = (conv['_merge'] == 'both').astype(int)

In [129]:
conv

,customer_id,offer_id,_merge,converted
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,both,1
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7,left_only,0
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,left_only,0
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4,left_only,0
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0,left_only,0
...,...,...,...,...
63283,670626b55bfb4ba39c85b27cc7cca527,0b1e1539f2cc45b7b9fa7c272da2e1d7,both,1
63284,a57890c3bbb7463e9018abb7fecadb15,5a8bc65990b245e5a138643cd4eb9837,left_only,0
63285,d087c473b4d247ccb0abfef59ba12b0e,ae264e3637204a6fb9bb56bc8210ddfd,both,1
63286,6d5f3a774f3d4714ab0c092238f3a1d7,2298d6c36e964ae4a3e7e9706d1fb8c2,left_only,0


1이랑 0으로 나눈상태에서 전체 전환율 구하기

In [130]:
converted_mean = conv['converted'].mean()

In [131]:
print('전체 전환율:', round(converted_mean * 100, 2), '%')

전체 전환율: 45.82 %


확인결과 프로모션을 받은 고객 중 약 45.82%가 완료까지 이어졌다는 결과가 나옴

100명이 프로모션을 받으면 46명은 완료한다는 말

프로모션 수신 대비 약 45.82%가 완료로 이어졌으며, 나머지 약 54%는 완료 단계까지 도달하지 못한 것으로 나타났다(기간 제한 없이 완료한 비율)

In [132]:
conv['converted'].value_counts()

converted
0    34292
1    28996
Name: count, dtype: int64

In [133]:
# 비율로 봤을 때
conv['converted'].value_counts(normalize=True)

converted
0    0.54184
1    0.45816
Name: proportion, dtype: float64

- 일단 지금까지의 결과는 기간이 어떻든 다포함된 값이니깐 프로모션 효과라고 보기에는 한계가 있음...
- duration(프로모션 기간)으로 이기간에 완료했는지 추가 분석

duration기준으로 전환율을 구해야되는데....

In [134]:
df2['duration'].value_counts()

duration
7.0     69969
5.0     37289
10.0    33860
3.0     14305
4.0     11761
Name: count, dtype: int64

In [135]:
df2['duration'].describe()

count    167184.000000
mean          6.608210
std           2.136267
min           3.000000
25%           5.000000
50%           7.000000
75%           7.000000
max          10.000000
Name: duration, dtype: float64

In [136]:
# 시간이랑 프로모션기간 추가해서 분리
duration_received = df2[df2['event'] == 'offer received'][['customer_id', 'offer_id', 'time', 'duration']]

In [137]:
# 완료는 언제 완료했는지 위해 시간컬럼까지 추가로 분리
duration_completed = df[df['event'] == 'offer completed'][['customer_id', 'offer_id', 'time']]

In [138]:
duration_received = df2[df2['event'] == 'offer received'][['customer_id', 'offer_id', 'time', 'duration']].drop_duplicates()
duration_completed = df2[df2['event'] == 'offer completed'][['customer_id', 'offer_id', 'time']].drop_duplicates()

In [139]:
duration_received.duplicated().sum()

np.int64(0)

In [140]:
duration_completed.duplicated().sum()

np.int64(0)

In [141]:
# merge할때 똑같은 시간컬럼이 있어 충돌 방지
duration_received = duration_received.rename(columns={'time': 'time_rec'})
duration_completed = duration_completed.rename(columns={'time': 'time_comp'})

In [142]:
# 고객 오퍼 기준 merge
conv_dur = duration_received.merge(
    duration_completed,
    on=['customer_id', 'offer_id'],
    how='left'
)

In [143]:
conv_dur

,customer_id,offer_id,time_rec,duration,time_comp
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,7.0,132.0
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,10.0,NaN
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,0,7.0,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4,0,10.0,NaN
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0,0,5.0,NaN
...,...,...,...,...,...
85515,d087c473b4d247ccb0abfef59ba12b0e,ae264e3637204a6fb9bb56bc8210ddfd,576,7.0,636.0
85516,cb23b66c56f64b109d673d5e56574529,2906b810c7d4411798c6938adc9daaa5,576,7.0,156.0
85517,6d5f3a774f3d4714ab0c092238f3a1d7,2298d6c36e964ae4a3e7e9706d1fb8c2,576,7.0,NaN
85518,9dc1421481194dcd9400aec7c9ae6366,ae264e3637204a6fb9bb56bc8210ddfd,576,7.0,594.0


시간기준 완료 컬럼 결측?? 완료안한사람이 결측처리 된건가??

In [144]:
# 그래서 수신 대비 완료까지의 경과 시간
conv_dur['time_diff'] = conv_dur['time_comp'] - conv_dur['time_rec']

In [145]:
conv_dur

,customer_id,offer_id,time_rec,duration,time_comp,time_diff
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,7.0,132.0,132.0
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,10.0,NaN,NaN
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,0,7.0,NaN,NaN
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4,0,10.0,NaN,NaN
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0,0,5.0,NaN,NaN
...,...,...,...,...,...,...
85515,d087c473b4d247ccb0abfef59ba12b0e,ae264e3637204a6fb9bb56bc8210ddfd,576,7.0,636.0,60.0
85516,cb23b66c56f64b109d673d5e56574529,2906b810c7d4411798c6938adc9daaa5,576,7.0,156.0,-420.0
85517,6d5f3a774f3d4714ab0c092238f3a1d7,2298d6c36e964ae4a3e7e9706d1fb8c2,576,7.0,NaN,NaN
85518,9dc1421481194dcd9400aec7c9ae6366,ae264e3637204a6fb9bb56bc8210ddfd,576,7.0,594.0,18.0


In [146]:
# 프로모션 기간안에 완료했으면 1 안했으면 0으로 컬럼생성(1일기준)
conv_dur['converted'] = (
    (conv_dur['time_diff'] <= conv_dur['duration'] * 24)
).fillna(False).astype(int)

In [147]:
conv_dur

,customer_id,offer_id,time_rec,duration,time_comp,time_diff,converted
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,7.0,132.0,132.0,1
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,10.0,NaN,NaN,0
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,0,7.0,NaN,NaN,0
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4,0,10.0,NaN,NaN,0
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0,0,5.0,NaN,NaN,0
...,...,...,...,...,...,...,...
85515,d087c473b4d247ccb0abfef59ba12b0e,ae264e3637204a6fb9bb56bc8210ddfd,576,7.0,636.0,60.0,1
85516,cb23b66c56f64b109d673d5e56574529,2906b810c7d4411798c6938adc9daaa5,576,7.0,156.0,-420.0,1
85517,6d5f3a774f3d4714ab0c092238f3a1d7,2298d6c36e964ae4a3e7e9706d1fb8c2,576,7.0,NaN,NaN,0
85518,9dc1421481194dcd9400aec7c9ae6366,ae264e3637204a6fb9bb56bc8210ddfd,576,7.0,594.0,18.0,1


오퍼 수신 시점부터 완료 시점까지의 경과 시간을 계산하여, 해당 완료가 오퍼 유효기간 내에 발생했는지를 기준으로 전환 여부를 판단(오퍼 유효기간(duration)이 일 단위로 제공)

# 마지막 duration기준 전환율!!

In [148]:
conversion_rate_dur = conv_dur['converted'].mean()
print('프로모션 유효기간 전환율:', round(conversion_rate_dur * 100, 2),'%')

프로모션 유효기간 전환율: 47.01 %


?? 전체 전환율보다 왜 더 높은값이 나오는거지;;;

아 중복제거 안했다;;;

아 뭐야 똑같은데??

In [149]:
(conv_dur['time_comp'] < conv_dur['time_rec']).sum()

np.int64(5630)

잘못된 데이터 행이 5630개가 있어서 전환율값이 이상하게 나왔다;;

In [ ]:
# 5630개 제거후 다시 전환율 계산
conv_dur['converted'] = (
    (conv_dur['time_diff'] >= 0) &
    (conv_dur['time_diff'] <= conv_dur['duration'] * 24)
).fillna(False).astype(int)

In [152]:
conversion_rate_dur = conv_dur['converted'].mean()
print('프로모션 유효기간 다시 전환율 확인', round(conversion_rate_dur * 100, 2),'%')

프로모션 유효기간 다시 전환율 확인 40.43 %


결론: 대부분의 전환은 프로모션 기간 내에 발생(값이 크게 차이가 안남..)

프로모션 효과 자체는 꽤 유효하며, 일부 고객은 지연 반응을 보임

전체 전환율은 45.82%로 나타났으며, 오퍼 유효기간을 반영한 전환율은 40.43%로 확인되었다. 이는 약 5%p 수준의 고객이 프로모션 기간 이후에 완료를 수행한 것으로 해석

프로모션은 실제로 효과가 있었던거 같음

# 이제 offer_type별 전환율 분석
- 어떤 프로모션이 고객한테 잘 통할까??

In [159]:
df2.head()

,customer_id,event,time,offer_id,amount,event_reward,offer_type,offer_reward,difficulty,duration,channels,web,email,mobile,social,gender,age,income,became_member_on,profile_missing
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,bogo,5.0,5.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,F,75.0,100000.0,2017-05-09,1
1,a03223e636434f42ac4c3df47e8bac43,offer received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,discount,5.0,20.0,10.0,"['web', 'email']",1.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,0
2,e2127556f4f64592b11af22de27a7932,offer received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,discount,2.0,10.0,7.0,"['web', 'email', 'mobile']",1.0,1.0,1.0,0.0,M,68.0,70000.0,2018-04-26,1
3,8ec6ce2a7e7949b1bf142def7d0e0586,offer received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,discount,2.0,10.0,10.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0
4,68617ca6246f4fbc85e91a2a49552598,offer received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,bogo,10.0,10.0,5.0,"['web', 'email', 'mobile', 'social']",1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,0


In [153]:
df2.columns

Index(['customer_id', 'event', 'time', 'offer_id', 'amount', 'event_reward',
       'offer_type', 'offer_reward', 'difficulty', 'duration', 'channels',
       'web', 'email', 'mobile', 'social', 'gender', 'age', 'income',
       'became_member_on', 'profile_missing'],
      dtype='str')

In [155]:
df2['offer_type'].duplicated().sum()

np.int64(306133)

In [156]:
df2['offer_id'].duplicated().sum()

np.int64(306126)

In [157]:
df2['offer_type'].nunique()

3

In [158]:
df2['offer_id'].nunique()

10

아... 같은 프로모션이 여러고객/이벤트에서 반복되서 중복으로 나타난다고 이벤트 자제의 특성???이라고 하면 되나?

In [160]:
# 분석할때는 offer_id 기준으로 프로모션 정보를 별도 분리
offer_info = df2[['offer_id', 'offer_type']].drop_duplicates()

In [161]:
offer_info

,offer_id,offer_type
0,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo
1,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount
2,2906b810c7d4411798c6938adc9daaa5,discount
3,fafdcd668e3743c1bb461111dcafc2a4,discount
4,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo
5,f19421c1d4aa40978ebb69ca19b0e20d,bogo
6,2298d6c36e964ae4a3e7e9706d1fb8c2,discount
7,3f207df678b143eea3cee63160fa8bed,informational
12,ae264e3637204a6fb9bb56bc8210ddfd,bogo
31,5a8bc65990b245e5a138643cd4eb9837,informational


In [162]:
offer_info.tail()

,offer_id,offer_type
6,2298d6c36e964ae4a3e7e9706d1fb8c2,discount
7,3f207df678b143eea3cee63160fa8bed,informational
12,ae264e3637204a6fb9bb56bc8210ddfd,bogo
31,5a8bc65990b245e5a138643cd4eb9837,informational
12654,NaN,NaN


In [164]:
offer_info.isna().sum()

offer_id      1
offer_type    1
dtype: int64

offer_id와 offer_type이 모두 결측인 행이 1건 확인되었는데, 이는 오퍼 관련 이벤트가 아닌 로그로 판단되어 분석 대상에서 제외

In [165]:
offer_info = df2[['offer_id', 'offer_type']].dropna().drop_duplicates()

In [166]:
offer_info.isna().sum()

offer_id      0
offer_type    0
dtype: int64

In [167]:
offer_info

,offer_id,offer_type
0,9b98b8c7a33c4b65b9aebfe6a799e6d9,bogo
1,0b1e1539f2cc45b7b9fa7c272da2e1d7,discount
2,2906b810c7d4411798c6938adc9daaa5,discount
3,fafdcd668e3743c1bb461111dcafc2a4,discount
4,4d5c57ea9a6940dd891ad53e9dbe8da0,bogo
5,f19421c1d4aa40978ebb69ca19b0e20d,bogo
6,2298d6c36e964ae4a3e7e9706d1fb8c2,discount
7,3f207df678b143eea3cee63160fa8bed,informational
12,ae264e3637204a6fb9bb56bc8210ddfd,bogo
31,5a8bc65990b245e5a138643cd4eb9837,informational


이제 type별 duration전환율 계산

In [168]:
conv_dur = conv_dur.merge(
    offer_info,
    on='offer_id',
    how='left'
)

In [169]:
conv_dur

,customer_id,offer_id,time_rec,duration,time_comp,time_diff,converted,offer_type
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,7.0,132.0,132.0,1,bogo
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,10.0,NaN,NaN,0,discount
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,0,7.0,NaN,NaN,0,discount
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4,0,10.0,NaN,NaN,0,discount
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0,0,5.0,NaN,NaN,0,bogo
...,...,...,...,...,...,...,...,...
85515,d087c473b4d247ccb0abfef59ba12b0e,ae264e3637204a6fb9bb56bc8210ddfd,576,7.0,636.0,60.0,1,bogo
85516,cb23b66c56f64b109d673d5e56574529,2906b810c7d4411798c6938adc9daaa5,576,7.0,156.0,-420.0,0,discount
85517,6d5f3a774f3d4714ab0c092238f3a1d7,2298d6c36e964ae4a3e7e9706d1fb8c2,576,7.0,NaN,NaN,0,discount
85518,9dc1421481194dcd9400aec7c9ae6366,ae264e3637204a6fb9bb56bc8210ddfd,576,7.0,594.0,18.0,1,bogo


In [170]:
offer_conv = conv_dur.groupby('offer_type')['converted'].mean().sort_values(ascending=False)

offer_conv

offer_type
discount         0.524051
bogo             0.459105
informational    0.000000
Name: converted, dtype: float64

offer_type별 전환율을 비교한 결과, bogo 및 discount 유형의 프로모션이 높은 전환율을 보였으며, informational 유형은 상대적으로 낮은 전환율을 나타냈다. 이는 금전적 보상이 포함된 프로모션이 고객 행동 유도에 더 효과적

informational 유형의 프로모션은 완료 이벤트가 존재하지 않는 구조이기 때문에 전환율이 0으로 나타났으며, 이는 해당 유형이 행동 유도보다는 정보 전달 목적의 프로모션임을 의미 따라서 informational 유형은 전환율 비교 분석에서 제외하거나 별도로 해석하는 것이 필요

# 채널별 분석

다시 일단 현재 까지 구한conv_dur에 채널 merge

In [171]:
df2.columns

Index(['customer_id', 'event', 'time', 'offer_id', 'amount', 'event_reward',
       'offer_type', 'offer_reward', 'difficulty', 'duration', 'channels',
       'web', 'email', 'mobile', 'social', 'gender', 'age', 'income',
       'became_member_on', 'profile_missing'],
      dtype='str')

In [172]:
df2['channels'].value_counts()

channels
['web', 'email', 'mobile', 'social']    77573
['web', 'email', 'mobile']              43626
['email', 'mobile', 'social']           32314
['web', 'email']                        13671
Name: count, dtype: int64

일단 채널중 거의 풀채널 여러채널이 동시에 사용되고 단일 사용된 채널은 없음

현재 channels 컬럼은 여러 채널이 조합된 형태로 구성되어 있어, 해당 값을 그대로 기준으로 전환율을 비교할 경우 각 채널의 개별 영향력을 해석하기 어렵다고 판단

따라서 email, mobile, web, social과 같이 채널을 개별 변수로 분리하여, 각 채널이 포함된 경우의 전환율을 기준으로 분석을 진행

In [173]:
df2[['email', 'mobile', 'web', 'social']].head()

,email,mobile,web,social
0,1.0,1.0,1.0,0.0
1,1.0,0.0,1.0,0.0
2,1.0,1.0,1.0,0.0
3,1.0,1.0,1.0,1.0
4,1.0,1.0,1.0,1.0


In [ ]:
# offer_id + 채널정보 조합 불어오고 중복은 제거
channel_info = df2[['offer_id', 'email', 'mobile', 'web', 'social']].drop_duplicates()

In [175]:
channel_info

,offer_id,email,mobile,web,social
0,9b98b8c7a33c4b65b9aebfe6a799e6d9,1.0,1.0,1.0,0.0
1,0b1e1539f2cc45b7b9fa7c272da2e1d7,1.0,0.0,1.0,0.0
2,2906b810c7d4411798c6938adc9daaa5,1.0,1.0,1.0,0.0
3,fafdcd668e3743c1bb461111dcafc2a4,1.0,1.0,1.0,1.0
4,4d5c57ea9a6940dd891ad53e9dbe8da0,1.0,1.0,1.0,1.0
5,f19421c1d4aa40978ebb69ca19b0e20d,1.0,1.0,1.0,1.0
6,2298d6c36e964ae4a3e7e9706d1fb8c2,1.0,1.0,1.0,1.0
7,3f207df678b143eea3cee63160fa8bed,1.0,1.0,1.0,0.0
12,ae264e3637204a6fb9bb56bc8210ddfd,1.0,1.0,0.0,1.0
31,5a8bc65990b245e5a138643cd4eb9837,1.0,1.0,0.0,1.0


또 위에 처럼 nan이 붙음 

In [176]:
channel_info.isna().sum()

offer_id    1
email       1
mobile      1
web         1
social      1
dtype: int64

이벤트 컬럼 특성상 불어올때 offer_id가 없는 행이 포함되서 그런거 같음

df2는 이벤트 로그 데이터이기 때문에 transaction과 같이 오퍼와 무관한 이벤트가 포함되어 있으며, 이 경우 offer_id가 결측으로 나타나는 것을 확인

따라서 오퍼 기반 분석을 위해 offer_id가 존재하는 행만 필터링하여 채널 정보를 구성

In [177]:
channel_info = df2[['offer_id', 'email', 'mobile', 'web', 'social']].dropna(subset=['offer_id']).drop_duplicates()

제거 이유: 오퍼 기반 분석을 위해 offer_id가 존재하는 행만 필터링하여 채널 정보를 구성

In [178]:
channel_info

,offer_id,email,mobile,web,social
0,9b98b8c7a33c4b65b9aebfe6a799e6d9,1.0,1.0,1.0,0.0
1,0b1e1539f2cc45b7b9fa7c272da2e1d7,1.0,0.0,1.0,0.0
2,2906b810c7d4411798c6938adc9daaa5,1.0,1.0,1.0,0.0
3,fafdcd668e3743c1bb461111dcafc2a4,1.0,1.0,1.0,1.0
4,4d5c57ea9a6940dd891ad53e9dbe8da0,1.0,1.0,1.0,1.0
5,f19421c1d4aa40978ebb69ca19b0e20d,1.0,1.0,1.0,1.0
6,2298d6c36e964ae4a3e7e9706d1fb8c2,1.0,1.0,1.0,1.0
7,3f207df678b143eea3cee63160fa8bed,1.0,1.0,1.0,0.0
12,ae264e3637204a6fb9bb56bc8210ddfd,1.0,1.0,0.0,1.0
31,5a8bc65990b245e5a138643cd4eb9837,1.0,1.0,0.0,1.0


In [179]:
channel_info.isna().sum()

offer_id    0
email       0
mobile      0
web         0
social      0
dtype: int64

In [180]:
# conv_dur에 붙이기
conv_dur = conv_dur.merge(
    channel_info,
    on='offer_id',
    how='left'
)

In [181]:
conv_dur.head()

,customer_id,offer_id,time_rec,duration,time_comp,time_diff,converted,offer_type,email,mobile,web,social
0,78afa995795e4d85b5d9ceeca43f5fef,9b98b8c7a33c4b65b9aebfe6a799e6d9,0,7.0,132.0,132.0,1,bogo,1.0,1.0,1.0,0.0
1,a03223e636434f42ac4c3df47e8bac43,0b1e1539f2cc45b7b9fa7c272da2e1d7,0,10.0,NaN,NaN,0,discount,1.0,0.0,1.0,0.0
2,e2127556f4f64592b11af22de27a7932,2906b810c7d4411798c6938adc9daaa5,0,7.0,NaN,NaN,0,discount,1.0,1.0,1.0,0.0
3,8ec6ce2a7e7949b1bf142def7d0e0586,fafdcd668e3743c1bb461111dcafc2a4,0,10.0,NaN,NaN,0,discount,1.0,1.0,1.0,1.0
4,68617ca6246f4fbc85e91a2a49552598,4d5c57ea9a6940dd891ad53e9dbe8da0,0,5.0,NaN,NaN,0,bogo,1.0,1.0,1.0,1.0


채널별로 붙었고 id를 기준으로 이제 전환율 계산

In [182]:
# 총 4개라 반복문 활용
for ch in ['email', 'mobile', 'web', 'social']:
    print(ch, conv_dur[conv_dur[ch] == 1]['converted'].mean())

email 0.4043147801683817
mobile 0.40247051343601187
web 0.445259903493311
social 0.4324126507184406


- email: 40%
- mobile: 40%
- web: 44%
- social: 43%

오.... 생각했던거랑 다른결과과 mobile이 가장 높을 줄알았는데 예상외로 web이 가장높고 email이랑 비슷한 전환율을 나타내고 있음..

결론: duration 기준 전환율을 바탕으로 채널별 효과를 분석한 결과, web 채널이 포함된 오퍼에서 가장 높은 전환율을 보였으며 mobile과 email은 유사한 수준을 나타냈다.
이는 채널 단독 효과보다는 오퍼 특성과 채널 조합의 영향이 함께 반영된 결과로 해석